# 06 — NWIS Cross-Reference

Saves the downloaded NWIS California stream site file as parquet and cross-references it against the restored USGS metadata to find sites that appear in both datasets.

Matching uses two independent signals:
- **Coordinate proximity**: NWIS site within 0.1° (~10 km) of a restored site
- **Name token overlap**: shared meaningful words between NWIS `station_nm` and restored `watersource_name`

Sites matching on both signals are the strongest candidates for cross-validation.

Inputs:
- `data/analysis/dam_exploring/nwis_streams_ca.txt`
- `data/analysis/processed_metadata.parquet`

Outputs:
- `data/analysis/nwis_ca_streams.parquet` — full parsed NWIS file
- `data/analysis/dam_exploring/nwis_crossref_matches.csv` — matched site pairs with scores

In [1]:
import pandas as pd
import numpy as np
import re
from scipy.spatial import cKDTree
from pathlib import Path

root         = Path('../..')
nwis_txt     = root / 'data/analysis/dam_exploring/nwis_streams_ca.txt'
parquet_path = root / 'data/analysis/processed_metadata.parquet'
out_parquet  = root / 'data/analysis/nwis_ca_streams.parquet'
out_matches  = root / 'data/analysis/dam_exploring/nwis_crossref_matches.csv'

## Load and save NWIS file

In [2]:
nwis_raw = pd.read_csv(
    nwis_txt,
    sep='\t',
    comment='#',
    low_memory=False,
    skiprows=[1],          # drop the format-descriptor row (5s, 15s, ...)
)

# Coerce numeric columns
for col in ('dec_lat_va', 'dec_long_va', 'alt_va', 'count_nu'):
    nwis_raw[col] = pd.to_numeric(nwis_raw[col], errors='coerce')
nwis_raw['begin_date'] = pd.to_datetime(nwis_raw['begin_date'], errors='coerce')
nwis_raw['end_date']   = pd.to_datetime(nwis_raw['end_date'],   errors='coerce')
nwis_raw['begin_year'] = nwis_raw['begin_date'].dt.year.astype('Int64')
nwis_raw['end_year']   = nwis_raw['end_date'].dt.year.astype('Int64')

nwis_raw.to_parquet(out_parquet, index=False)
print(f'Saved full NWIS file: {out_parquet}')
print(f'  {len(nwis_raw):,} rows  |  {nwis_raw["site_no"].nunique():,} unique sites')
print()
print('data_type_cd breakdown:')
print(nwis_raw['data_type_cd'].value_counts().to_string())
print()
print('Top parameter codes (parm_cd):')
parm_labels = {'00060': 'discharge (cfs)', '00065': 'gage height', '00010': 'water temp',
               '00095': 'specific conductance', '63160': 'water surface elevation'}
for code, cnt in nwis_raw['parm_cd'].value_counts().head(8).items():
    print(f'  {code}  {parm_labels.get(str(code), "")}  —  {cnt:,} records')

Saved full NWIS file: ..\..\data\analysis\nwis_ca_streams.parquet
  9,821 rows  |  2,981 unique sites

data_type_cd breakdown:
data_type_cd
dv    4884
uv    2804
pk    2132
2s       1

Top parameter codes (parm_cd):
  00060  discharge (cfs)  —  3,080 records
  00010  water temp  —  1,350 records
  00065  gage height  —  703 records
  00095  specific conductance  —  494 records
  63160  water surface elevation  —  440 records
  63680    —  311 records
  80155    —  223 records
  80154    —  218 records


C:\Users\aeliz\AppData\Local\Temp\ipykernel_15836\92920330.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  nwis_raw['begin_date'] = pd.to_datetime(nwis_raw['begin_date'], errors='coerce')
C:\Users\aeliz\AppData\Local\Temp\ipykernel_15836\92920330.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  nwis_raw['end_date']   = pd.to_datetime(nwis_raw['end_date'],   errors='coerce')


## Build unique NWIS discharge site table

Keep one row per site using daily discharge records (`dv` + `parm_cd=00060`). Where a site has multiple `dv/00060` rows (rare — happens for revised or split records), keep the one with the earliest begin date.

In [3]:
dv_q = nwis_raw[
    (nwis_raw['data_type_cd'] == 'dv') &
    (nwis_raw['parm_cd'].astype(str) == '00060') &
    nwis_raw['dec_lat_va'].notna() &
    nwis_raw['dec_long_va'].notna()
].copy()

# One row per site: keep earliest begin_date
nwis_sites = (
    dv_q.sort_values('begin_date')
    .groupby('site_no', as_index=False)
    .agg(
        station_nm   = ('station_nm',   'first'),
        lat          = ('dec_lat_va',   'first'),
        lon          = ('dec_long_va',  'first'),
        huc_cd       = ('huc_cd',       'first'),
        begin_year   = ('begin_year',   'min'),
        end_year     = ('end_year',     'max'),
        total_days   = ('count_nu',     'sum'),
    )
)

print(f'Unique NWIS discharge sites (dv/00060, with coords): {len(nwis_sites):,}')
print()
print('Period of record distribution:')
bins = [0, 1900, 1920, 1940, 1960, 1980, 2030]
labels = ['pre-1900', '1900–1919', '1920–1939', '1940–1959', '1960–1979', '1980+']
nwis_sites['begin_era'] = pd.cut(nwis_sites['begin_year'], bins=bins, labels=labels, right=False)
print(nwis_sites['begin_era'].value_counts().reindex(labels).to_string())

Unique NWIS discharge sites (dv/00060, with coords): 2,418

Period of record distribution:
begin_era
pre-1900      15
1900–1919    241
1920–1939    336
1940–1959    460
1960–1979    697
1980+        669


## Load restored metadata — unique stream discharge sites

Same deduplication as `05_predam_gauge_map.ipynb`: group by rounded coordinates + normalized name.

In [4]:
CA_LAT = (32.5, 42.1)
CA_LON = (-124.6, -114.1)

df = pd.read_parquet(parquet_path)

sd = df[
    (df['water_type_clean'] == 'Stream Discharge') &
    df['lat_combined'].notna() &
    df['lon_combined'].notna() &
    df['year_start'].notna() &
    df['lat_combined'].between(*CA_LAT) &
    df['lon_combined'].between(*CA_LON)
].copy()

sd['lat_r'] = sd['lat_combined'].round(2)
sd['lon_r'] = sd['lon_combined'].round(2)
sd['source_norm'] = sd['watersource_name'].fillna('').str.lower().str.strip()

restored = (
    sd.groupby(['lat_r', 'lon_r', 'source_norm'], as_index=False)
    .agg(
        watersource_name = ('watersource_name', 'first'),
        year_start       = ('year_start',        'min'),
        year_end         = ('year_end',           'max'),
        coord_source     = ('coord_source',       'first'),
        n_docs           = ('id',                 'nunique'),
    )
)

print(f'Restored unique stream discharge sites: {len(restored):,}')

Restored unique stream discharge sites: 34,692


## Signal 1 — Coordinate proximity match

For each NWIS site, find all restored sites within 0.1° (~10 km). Uses a KDTree for efficiency.

In [5]:
# Build KDTree on restored site coords
restored_coords = restored[['lat_r', 'lon_r']].values
tree = cKDTree(restored_coords)

nwis_coords = nwis_sites[['lat', 'lon']].values

COORD_THRESH = 0.1   # degrees (~10 km)
distances, indices = tree.query(nwis_coords, k=5, distance_upper_bound=COORD_THRESH)

# Build a flat match table: one row per (nwis_site, restored_site) pair within threshold
coord_matches = []
n_restored = len(restored)

for i, (dists, idxs) in enumerate(zip(distances, indices)):
    nrow = nwis_sites.iloc[i]
    for dist, idx in zip(dists, idxs):
        if idx < n_restored:   # cKDTree returns n_restored for "no match"
            rrow = restored.iloc[idx]
            coord_matches.append({
                'nwis_site_no':      nrow['site_no'],
                'nwis_name':         nrow['station_nm'],
                'nwis_lat':          nrow['lat'],
                'nwis_lon':          nrow['lon'],
                'nwis_begin_year':   nrow['begin_year'],
                'nwis_end_year':     nrow['end_year'],
                'restored_name':     rrow['watersource_name'],
                'restored_lat':      rrow['lat_r'],
                'restored_lon':      rrow['lon_r'],
                'restored_year_start': rrow['year_start'],
                'restored_year_end': rrow['year_end'],
                'coord_source':      rrow['coord_source'],
                'n_docs':            rrow['n_docs'],
                'coord_dist_deg':    round(dist, 4),
            })

coord_df = pd.DataFrame(coord_matches)
print(f'NWIS sites with at least one coord match (≤0.1°): '
      f"{coord_df['nwis_site_no'].nunique():,} of {len(nwis_sites):,}")
print(f'Total (nwis, restored) coordinate pairs: {len(coord_df):,}')

NWIS sites with at least one coord match (≤0.1°): 2,407 of 2,418
Total (nwis, restored) coordinate pairs: 11,822


## Signal 2 — Name token overlap

Normalize both names by lowercasing, removing punctuation, and stripping common geographic stop-words. Score = number of shared meaningful tokens.

In [6]:
STOP = {
    'river', 'creek', 'stream', 'near', 'at', 'above', 'below', 'nr', 'ab', 'bl',
    'ca', 'calif', 'california', 'fork', 'branch', 'north', 'south', 'east', 'west',
    'middle', 'upper', 'lower', 'the', 'of', 'and', 'in', 'on', 'a', 'an',
    'gage', 'station', 'site', 'tributary', 'trib', 'run', 'wash', 'gulch', 'slough',
}

def tokenize(name):
    name = re.sub(r'[^a-z0-9\s]', ' ', str(name).lower())
    return {t for t in name.split() if t not in STOP and len(t) > 1}

nwis_sites['tokens'] = nwis_sites['station_nm'].apply(tokenize)
coord_df['nwis_tokens']     = coord_df['nwis_name'].apply(tokenize)
coord_df['restored_tokens'] = coord_df['restored_name'].apply(tokenize)

coord_df['shared_tokens'] = coord_df.apply(
    lambda r: r['nwis_tokens'] & r['restored_tokens'], axis=1
)
coord_df['name_score'] = coord_df['shared_tokens'].apply(len)
coord_df['shared_token_str'] = coord_df['shared_tokens'].apply(lambda s: ', '.join(sorted(s)))

# Drop the set columns — not CSV-serializable
coord_df = coord_df.drop(columns=['nwis_tokens', 'restored_tokens', 'shared_tokens'])

print('Name score distribution (shared tokens, coord-matched pairs):')
print(coord_df['name_score'].value_counts().sort_index().to_string())

Name score distribution (shared tokens, coord-matched pairs):
name_score
0    4551
1    3186
2    2541
3    1146
4     319
5      69
6      10


## Combine signals and flag match quality

| Tier | Criteria |
|------|----------|
| **Strong** | coord ≤ 0.05° AND name_score ≥ 2 |
| **Coord only** | coord ≤ 0.1° AND name_score < 2 |
| **Name only** | name_score ≥ 3 (no coord constraint — separate pass) |

In [7]:
def match_tier(row):
    if row['coord_dist_deg'] <= 0.05 and row['name_score'] >= 2:
        return 'strong'
    elif row['name_score'] >= 2:
        return 'coord+name'
    else:
        return 'coord_only'

coord_df['match_tier'] = coord_df.apply(match_tier, axis=1)

# Keep best match per NWIS site (highest name_score, then closest distance)
best = (
    coord_df
    .sort_values(['name_score', 'coord_dist_deg'], ascending=[False, True])
    .groupby('nwis_site_no', as_index=False)
    .first()
)

print('Match tier breakdown (best match per NWIS site):')
print(best['match_tier'].value_counts().to_string())
print()
print(f'Strong matches (coord ≤0.05° + name ≥2 tokens): {(best["match_tier"]=="strong").sum():,}')
print()

# Pre-1920 NWIS sites with a strong or coord+name match
early = best[best['nwis_begin_year'] < 1920]
print(f'Pre-1920 NWIS sites: {len(early):,}')
print(f'  with strong match:       {(early["match_tier"]=="strong").sum():,}')
print(f'  with coord+name match:   {(early["match_tier"]=="coord+name").sum():,}')
print(f'  with coord_only match:   {(early["match_tier"]=="coord_only").sum():,}')

Match tier breakdown (best match per NWIS site):
match_tier
strong        1646
coord_only     731
coord+name      30

Strong matches (coord ≤0.05° + name ≥2 tokens): 1,646

Pre-1920 NWIS sites: 255
  with strong match:       212
  with coord+name match:   4
  with coord_only match:   39


## Strong matches — sample

In [8]:
strong = best[best['match_tier'] == 'strong'].copy()
strong['overlap_start'] = strong[['nwis_begin_year', 'restored_year_start']].max(axis=1)
strong['overlap_end']   = strong[['nwis_end_year',   'restored_year_end']].min(axis=1)
strong['overlap_years'] = (strong['overlap_end'] - strong['overlap_start']).clip(lower=0)

display_cols = ['nwis_site_no', 'nwis_name', 'nwis_begin_year', 'nwis_end_year',
                'restored_name', 'restored_year_start', 'restored_year_end',
                'overlap_years', 'coord_dist_deg', 'name_score', 'shared_token_str']

print(f'Strong matches: {len(strong):,}  (showing top 20 by overlap years)')
with pd.option_context('display.max_colwidth', 40, 'display.width', 200):
    print(strong[display_cols].sort_values('overlap_years', ascending=False).head(20).to_string(index=False))

Strong matches: 1,646  (showing top 20 by overlap years)
nwis_site_no                                   nwis_name  nwis_begin_year  nwis_end_year                                              restored_name  restored_year_start  restored_year_end  overlap_years  coord_dist_deg  name_score                     shared_token_str
    11179000                       ALAMEDA C NR NILES CA             1891           2026                           ALAMEDA CREEK NEAR NILES, CALIF.                 1891               1975             84          0.0030           2                       alameda, niles
    11377100 SACRAMENTO R AB BEND BRIDGE NR RED BLUFF CA             1891           2026 SACRAMENTO RIVER ABOVE BEND BRIDGE, NEAR RED BLUFF, CALIF.                 1879               1975             84          0.0037           5 bend, bluff, bridge, red, sacramento
    11051500                   SANTA ANA R NR MENTONE CA             1896           2026                               Santa Ana River near

## Pre-1920 NWIS sites — how many have any restored match?

In [9]:
pre1920_nwis = nwis_sites[nwis_sites['begin_year'] < 1920].copy()
matched_ids  = set(best['nwis_site_no'])

pre1920_nwis['has_restored_match'] = pre1920_nwis['site_no'].isin(matched_ids)

print(f'Pre-1920 NWIS sites total:        {len(pre1920_nwis):,}')
print(f'  with any coord match (≤0.1°):   {pre1920_nwis["has_restored_match"].sum():,}')
print(f'  no coord match found:            {(~pre1920_nwis["has_restored_match"]).sum():,}')
print()

# Show unmatched pre-1920 NWIS sites — interesting gaps
unmatched = pre1920_nwis[~pre1920_nwis['has_restored_match']]
print('Unmatched pre-1920 NWIS sites (gauges with no nearby restored record):')
with pd.option_context('display.max_colwidth', 50, 'display.width', 160):
    print(unmatched[['site_no', 'station_nm', 'begin_year', 'end_year', 'lat', 'lon']]
          .sort_values('begin_year').to_string(index=False))

Pre-1920 NWIS sites total:        256
  with any coord match (≤0.1°):   255
  no coord match found:            1

Unmatched pre-1920 NWIS sites (gauges with no nearby restored record):
 site_no                 station_nm  begin_year  end_year       lat         lon
10354000 LONG VALLEY C NR SCOTTS CA        1917      1994 39.855461 -120.067702


## Save match table

In [10]:
# Save full match table (all coord pairs) and the best-per-site summary
coord_df.to_csv(out_matches, index=False)
best_path = out_matches.parent / 'nwis_crossref_best_match.csv'
best.to_csv(best_path, index=False)

print(f'Saved all pairs:  {out_matches}  ({len(coord_df):,} rows)')
print(f'Saved best match: {best_path}  ({len(best):,} rows)')
print()
print('Summary:')
print(f'  NWIS discharge sites:          {len(nwis_sites):,}')
print(f'  Restored sites:                {len(restored):,}')
print(f'  NWIS with any coord match:     {coord_df["nwis_site_no"].nunique():,}')
print(f'  Strong matches (both signals): {(best["match_tier"]=="strong").sum():,}')
print(f'  Pre-1920 NWIS w/ coord match:  {pre1920_nwis["has_restored_match"].sum():,} / {len(pre1920_nwis):,}')

Saved all pairs:  ..\..\data\analysis\dam_exploring\nwis_crossref_matches.csv  (11,822 rows)
Saved best match: ..\..\data\analysis\dam_exploring\nwis_crossref_best_match.csv  (2,407 rows)

Summary:
  NWIS discharge sites:          2,418
  Restored sites:                34,692
  NWIS with any coord match:     2,407
  Strong matches (both signals): 1,646
  Pre-1920 NWIS w/ coord match:  255 / 256
